In [ ]:
import sys
import os
from pathlib import Path

# Find repo root
repo_root = Path('.').resolve()
for _ in range(5):
    if (repo_root / 'src').is_dir():
        repo_root = f"{repo_root}/src"
        break
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root))

print(f"Repo root: {repo_root}")

from pprint import pprint
from iceye import iceye_v2
from shared_utils import s3utils
import json

In [ ]:
# Activation event metadata
EVENT_NAME = '202406_Example_Event'    # YYYYMM_Hazard_Location
SOURCE = "CSDA"  # Data origin (e.g., USGS, Copernicus, CSDA, TBD)

###### CONFIG #######
bucket = "csdap-iceye-delivery"
prefix = "disasters"
DATE = "2026-10-09 04:05:55"   # "YYYY-MM-DD HH:MM:SS" -- the closest GRD is used
FILTER_SIZE = 5                # 3, 5 or 7
OUTPUT_DIR = "/tmp/iceye_testing"  # emptied by reset_output_dir below; must sit under /tmp

# Target CRS for the output COG.
# None preserves the source projection (fastest, no warp).
# Uncomment the EPSG:3857 line if you plan to push the COG through
# veda-data-airflow's build_stac, which trips on the EPSG:4326 ensemble.
TARGET_CRS = None
# TARGET_CRS = "EPSG:3857"

In [ ]:
from shared_utils import PROCESSOR_STRING
ACTIVATION_METADATA = {
    "ACTIVATION_EVENT": EVENT_NAME,
    "SOURCE": SOURCE,
    "PROCESSOR": PROCESSOR_STRING,
    # Add any custom key-value pairs here
}


In [ ]:
# Show available capture dates for this sensor in S3 so the operator
# can cross-check what's passed to retrieve_iceye_resources below. (Issue #15)
from shared_utils.s3utils import retrieve_s3_valid_dates

available_dates = retrieve_s3_valid_dates(bucket, prefix)
print(f"{len(available_dates)} capture dates available in s3://{bucket}/{prefix}/")
pprint(available_dates)


In [ ]:
metadata, tifs = iceye_v2.retrieve_iceye_resources(DATE, bucket, prefix)

In [ ]:
# Start from an EMPTY OUTPUT_DIR so this run holds only its own product, and
# download the raw GRD + XML into a per-run scratch dir that is deleted when the
# block exits (success or failure) -- never the shared /tmp/s3_temp.
from shared_utils import reset_output_dir
from shared_utils.scratch import download_scratch

reset_output_dir(OUTPUT_DIR)
with download_scratch("iceye") as _dl:
    saved_product = iceye_v2.sigmaCalib(
        tifs, metadata,
        save_location=OUTPUT_DIR,
        filter_size=FILTER_SIZE,
        download_dir=_dl,
    )
saved_product  # <OUTPUT_DIR>/Backscatter/<ICEYE-Xnn>_sigma0-dB_filtered<N>_<ISO-Zulu>.tif

In [ ]:
from shared_utils import cog_utils
cog_filepath = cog_utils.convert_to_cog(
    saved_product,
    nodata=iceye_v2.ICEYE_NODATA,
    dst_crs=TARGET_CRS,
    metadata=ACTIVATION_METADATA,
    compression_level=1,
)
print(cog_filepath)

In [ ]:
# Preview the output COG to sanity-check the processing visually.
# preview_cogs reads via rasterio with overview-aware downsampling so it
# stays fast even on multi-GB COGs.
from shared_utils.plotting import preview_cogs

preview_cogs([cog_filepath], sample_n=4)
